# Decompositions of Multi-Controlled X Gates

This study compares constructions of an $n$-controlled X gate for
$n = 3, \dots, 20$. Each construction is expanded into an explicit circuit,
verified on small cases, and lowered to Clifford+T with the fixed gadgets of
`logique.benchmarks.costs`. We report the ancillas a construction needs and
whether they must be clean, its Toffoli count and depth, its T-count and
T-depth, its CNOT count, and the arbitrary-angle rotations that remain after
lowering, which require approximate synthesis.

The constructions are those available in Qiskit 2.1.1 (Barenco et al.,
Iten et al., Maslov, Vale et al. and Khattar and Gidney), a Toffoli tree with
relative-phase uncomputation, and the throwaway-ancilla tree of the
manuscript's Figure 1. The throwaway tree leaves its ancillas in a garbage
state, so it is valid only when a mirrored gate later restores them; its costs
are therefore those of one half of a compute–uncompute pair. Two analytical
references complete the comparison: Theorem 1 of Dutta et al., and the
measurement-assisted logical-AND tree, which is reported separately because it
relies on measurement and feedforward.

## 1. Choose the output folder

The notebook runner supplies `LOGIQUE_OUTPUT`; an interactive run creates a new
folder under `dump/`.

In [ ]:
from pathlib import Path
import os

from logique.paths import new_dump_directory

ROOT = Path(os.environ.get("LOGIQUE_WORKSPACE", Path.cwd())).resolve()
if "LOGIQUE_OUTPUT" in os.environ:
    OUT = Path(os.environ["LOGIQUE_OUTPUT"]).resolve()
else:
    OUT = new_dump_directory("mcx-decompositions", root=ROOT / "dump")
print("Artifacts:", OUT)

## 2. Verify every restoring construction on small gates

For 3 and 4 controls, compare each expanded circuit with the ideal gate. Dirty
constructions must match on every state of the borrowed wires; clean
constructions must match whenever their appended ancillas start in $|0\rangle$.
The circuit contains $n-2$ spare wires that dirty constructions may borrow.

In [ ]:
import numpy as np
import pandas as pd
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator

from logique.benchmarks import costs


def verify(method: str, controls: int) -> bool:
    width = 2 * controls - 1
    circuit = QuantumCircuit(width)
    circuit.mcx(list(range(controls)), controls, ctrl_state=(1 << controls) - 2)
    expanded, info = costs.decompose_mcx(circuit, method)
    lowered = costs.clifford_t(expanded)
    padded = QuantumCircuit(lowered.num_qubits)
    padded.compose(circuit, qubits=range(width), inplace=True)
    columns = [i for i in range(2**lowered.num_qubits) if i >> width == 0]
    return np.allclose(
        Operator(lowered).data[:, columns], Operator(padded).data[:, columns]
    )


restoring = [
    name
    for name, item in costs.MCX_DECOMPOSITIONS.items()
    if item.workspace != "throwaway"
]
checks = pd.DataFrame(
    [
        {"method": method, "controls": n, "exact": verify(method, n)}
        for method in restoring
        for n in (3, 4)
    ]
)
assert checks["exact"].all()
display(checks.pivot(index="method", columns="controls", values="exact"))

## 3. Cost each construction

The Gray-code construction has an exponential gate count, so it is evaluated
only up to 10 controls. Qiskit expresses the constructions of Barenco et al.,
Iten et al., Vale et al. and the Gray code directly in Clifford+T gates and
phase rotations, so their Toffoli columns are zero; compare them by T-count,
T-depth and rotations.

In [ ]:
rows = []
for method in costs.MCX_DECOMPOSITIONS:
    largest = 10 if method == "gray_code" else 20
    for n in range(3, largest + 1):
        rows.append(costs.isolated_mcx_costs(method, n))
measured = pd.DataFrame(rows)
measured["metric_level"] = "explicit"
measured.to_csv(OUT / "mcx_decompositions.csv", index=False)
display(
    measured[measured["controls"].isin([4, 8, 12, 16, 20])]
    .set_index(["method", "controls"])
    .drop(columns=["reference", "metric_level"])
)

## 4. Add the analytical references

Theorem 1 of Dutta et al. applies from the smallest range of controls allowed by
each $m_1$. Its Example 1 reports a Toffoli depth of 14 for $n = 32$ and
$m_1 = 3$, whereas the theorem gives 16; the rows below follow the theorem.
The measurement-assisted logical-AND tree uses $n - 2$ clean ancillas,
$4n - 4$ T gates and a T-depth of $\lceil \log_2 n \rceil + 1$ (Dutta et al.,
Corollary 2, after Gidney); its uncomputation needs measurement and
feedforward, so it is not mixed with the static constructions.

In [ ]:
import math

analytical = []
for m1 in (2, 3, 4):
    for n in range(3, 21):
        try:
            analytical.append(costs.dutta_mcx_costs(n, m1))
        except ValueError:
            continue
for n in range(3, 21):
    analytical.append(
        {
            "method": "logical_and_tree",
            "controls": n,
            "workspace": "clean",
            "ancillas": n - 2,
            "toffoli_count": n - 1,
            "toffoli_depth": math.ceil(math.log2(n)),
            "t_count": 4 * n - 4,
            "t_depth": math.ceil(math.log2(n)) + 1,
            "reference": "Gidney logical-AND; Dutta et al., Corollary 2",
            "metric_level": "analytical, measurement-assisted",
        }
    )
analytical = pd.DataFrame(analytical)
analytical.to_csv(OUT / "mcx_analytical.csv", index=False)
display(
    analytical[analytical["controls"].isin([8, 12, 16, 20])]
    .set_index(["method", "controls"])
    .drop(columns=["reference"])
)

## 5. Plot the trade-offs

The figure shows eight representative constructions, each with one color in
every panel; the remaining constructions appear in the tables above. Dashed
lines are analytical. The throwaway tree is costed as built, that is, as one
half of a compute–uncompute pair whose mirror restores its ancillas. Rotations
appear only in some constructions and their T cost depends on the synthesis
precision, so they are listed in a table after the figure. Identical values
overlap: the Iten construction, both trees and the logical-AND tree all use
n − 2 ancillas, and the restoring tree has the same T-count as the one-clean
Khattar–Gidney construction.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

# Reference categorical palette in fixed order; each method keeps its color.
palette = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
shown = {
    "noaux_v24": "Vale et al., no ancilla",
    "1_clean_kg24": "Khattar–Gidney, 1 clean",
    "2_clean_kg24": "Khattar–Gidney, 2 clean",
    "n_dirty_i15": "Iten et al., n − 2 dirty",
    "clean_tree": "Restoring tree, n − 2 clean",
    "throwaway_tree": "Throwaway tree, n − 2 held",
    "dutta25_m1_3": "Dutta et al., m1 = 3 (Theorem 1)",
    "logical_and_tree": "Logical-AND tree (measurement-assisted)",
}
color = dict(zip(shown, palette))
combined = pd.concat([measured, analytical], ignore_index=True)
plt.rcParams.update({"figure.dpi": 110, "font.size": 9})
figure, axes = plt.subplots(1, 3, figsize=(12, 4.2), sharex=True)
panels = [("ancillas", "Ancillas", "linear"), ("t_count", "T-count", "log"), ("t_depth", "T-depth", "log")]
for ax, (column, label, scale) in zip(axes, panels):
    for name, title in shown.items():
        series = combined[combined["method"] == name].set_index("controls")[column]
        series = series.astype(float)
        if scale == "log":
            series = series.where(series > 0)  # a zero leaves a gap on a log axis
        ax.plot(
            series.index,
            series.values,
            color=color[name],
            linewidth=1.8,
            linestyle="--" if name in ("dutta25_m1_3", "logical_and_tree") else "-",
            marker="o",
            markersize=3.5,
            label=title,
        )
    ax.set_yscale(scale)
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))
    if scale == "linear":
        ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.set_title(label)
    ax.set_xlabel("Controls")
    ax.grid(color="#e4e3df", linewidth=0.8)
    ax.spines[["top", "right"]].set_visible(False)
handles, labels = axes[0].get_legend_handles_labels()
figure.legend(handles, labels, loc="lower center", ncol=4, frameon=False)
figure.tight_layout(rect=(0, 0.13, 1, 1))
figure.savefig(OUT / "mcx_decompositions.png", dpi=150)
figure.savefig(OUT / "mcx_decompositions.svg")
plt.show()

In [ ]:
rotations = measured.pivot_table(
    index="method", columns="controls", values="rotation_count"
)
rotations = rotations[rotations.sum(axis=1) > 0]
display(rotations[[n for n in (3, 4, 6, 8, 10, 12, 16, 20) if n in rotations]])

## 6. Record provenance

Save the package versions and the checkout so the tables can be traced to this
run.

In [ ]:
import json

from logique.benchmarks.provenance import checkout_revision, environment

manifest = {
    "controls": [3, 20],
    "gray_code_limit": 10,
    "constructions": {
        name: {"workspace": item.workspace, "reference": item.reference}
        for name, item in costs.MCX_DECOMPOSITIONS.items()
    },
    "clifford_t_rules": {
        "ccx": "7 T, T-depth 3",
        "rccx and controlled Rx(±pi)": "4 T, T-depth 2",
        "ch": "2 T",
        "rotations": "counted separately",
    },
    "versions": environment(),
    "checkout": checkout_revision(ROOT),
}
(OUT / "mcx_manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
print(sorted(path.name for path in OUT.iterdir()))